# Relative Positional Encodings (Shaw et al., 2018 → Transformer-XL, 2019)The key conceptual shift: stop encoding "I am at position 47" and start encoding"I am 3 tokens away from you."This notebook covers:1. Why absolute PE is fundamentally limited2. Shaw et al.'s relative position representations3. Transformer-XL's reformulation of relative attention4. How relative PE improves generalization (and where it still falls short)

In [ ]:
import torchimport torch.nn as nnimport torch.nn.functional as Fimport matplotlib.pyplot as pltimport numpy as np%matplotlib inline

## Part 1: The Core Insight — Why Relative Distance Matters MoreConsider these two sentences:- "The **cat** sat on the **mat**" (cat at pos 1, mat at pos 5, distance = 4)- "Yesterday the **cat** sat on the **mat**" (cat at pos 2, mat at pos 6, distance = 4)The relationship between "cat" and "mat" is the same in both — they're 4 tokens apart.But with absolute PE:- In sentence 1: the model sees (pos=1, pos=5)- In sentence 2: the model sees (pos=2, pos=6)These are **completely different** position pairs! The model has to independently learnthat (1,5), (2,6), (3,7), (100,104) all mean "4 tokens apart."With relative PE, all of these just become **distance = 4**. One thing to learn instead of thousands.

In [ ]:
# Let's visualize: absolute positions vs relative distancesseq_len = 8# Absolute: each pair (i, j) is a unique combinationabs_pairs = torch.zeros(seq_len, seq_len, 2)for i in range(seq_len):    for j in range(seq_len):        abs_pairs[i, j] = torch.tensor([i, j])# Relative: each pair (i, j) maps to a single number (i - j)rel_distances = torch.zeros(seq_len, seq_len)for i in range(seq_len):    for j in range(seq_len):        rel_distances[i, j] = i - jfig, axes = plt.subplots(1, 2, figsize=(12, 5))# Show the relative distance matrixim = axes[0].imshow(rel_distances.numpy(), cmap='RdBu_r', aspect='auto')axes[0].set_title('Relative Distance Matrix (i - j)', fontsize=12)axes[0].set_xlabel('Key position (j)')axes[0].set_ylabel('Query position (i)')for i in range(seq_len):    for j in range(seq_len):        axes[0].text(j, i, f'{int(rel_distances[i,j])}', ha='center', va='center', fontsize=9)plt.colorbar(im, ax=axes[0])# Show how many unique values each approach needsabs_unique = seq_len * seq_len  # every (i,j) pair is uniquerel_unique = 2 * seq_len - 1   # distances from -(seq_len-1) to +(seq_len-1)bars = axes[1].bar(['Absolute\n(unique pairs)', 'Relative\n(unique distances)'],                     [abs_unique, rel_unique], color=['#e74c3c', '#2ecc71'], width=0.5)axes[1].set_ylabel('Number of unique position encodings needed')axes[1].set_title('Absolute vs Relative: Complexity', fontsize=12)for bar, val in zip(bars, [abs_unique, rel_unique]):    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,                 str(val), ha='center', fontsize=12, fontweight='bold')plt.tight_layout()plt.show()print(f"For seq_len={seq_len}:")print(f"  Absolute PE: {abs_unique} unique position pairs to learn")print(f"  Relative PE: {rel_unique} unique distances to learn")print(f"  For seq_len=512: absolute needs 262,144 pairs, relative needs just 1,023 distances!")

## Part 2: Shaw et al. (2018) — Relative Position RepresentationsThe first paper to formalize relative PE in transformers. The idea:In standard attention, the score between query at position `i` and key at position `j` is:$$\text{score}(i, j) = q_i \cdot k_j$$Shaw et al. added a **relative position embedding** to the key:$$\text{score}(i, j) = q_i \cdot (k_j + r_{i-j})$$Where $r_{i-j}$ is a **learned embedding** for the relative distance `(i - j)`.They also modified the value computation similarly:$$\text{output}_i = \sum_j \alpha_{ij} (v_j + r^V_{i-j})$$The relative embeddings $r$ are clipped to a maximum distance $k$:- Distances beyond $[-k, k]$ all share the same embedding- This means the model says "anything more than k tokens away is equally far"Let's implement this:

In [ ]:
class ShawRelativeAttention(nn.Module):    """    Self-attention with relative position representations (Shaw et al., 2018).    """    def __init__(self, d_model, max_relative_distance=8):        super().__init__()        self.d_model = d_model        self.k = max_relative_distance  # clip distance to [-k, k]                self.W_q = nn.Linear(d_model, d_model, bias=False)        self.W_k = nn.Linear(d_model, d_model, bias=False)        self.W_v = nn.Linear(d_model, d_model, bias=False)                # Relative position embeddings: 2k + 1 possible distances        # Index 0 = distance -k, index k = distance 0, index 2k = distance +k        num_positions = 2 * max_relative_distance + 1        self.rel_emb_k = nn.Embedding(num_positions, d_model)  # for keys        self.rel_emb_v = nn.Embedding(num_positions, d_model)  # for values        def _get_relative_indices(self, seq_len):        """Build the (seq_len, seq_len) matrix of clipped relative distances."""        positions = torch.arange(seq_len)        # relative_dist[i, j] = i - j, clipped to [-k, k], then shifted to [0, 2k]        relative_dist = positions.unsqueeze(0) - positions.unsqueeze(1)  # (seq_len, seq_len)        relative_dist = relative_dist.clamp(-self.k, self.k)  # clip        relative_dist = relative_dist + self.k  # shift to non-negative indices        return relative_dist        def forward(self, x):        """x: (batch, seq_len, d_model)"""        batch, seq_len, _ = x.shape                Q = self.W_q(x)  # (batch, seq_len, d_model)        K = self.W_k(x)        V = self.W_v(x)                # Standard content-content attention        content_score = Q @ K.transpose(-2, -1)  # (batch, seq_len, seq_len)                # Relative position contribution to attention score        rel_indices = self._get_relative_indices(seq_len)  # (seq_len, seq_len)        rel_k = self.rel_emb_k(rel_indices)  # (seq_len, seq_len, d_model)                # q_i · r_{i-j} for all (i, j) pairs        # Q: (batch, seq_len, d_model) → need to dot with rel_k: (seq_len, seq_len, d_model)        rel_score = torch.einsum('bid,ijd->bij', Q, rel_k)  # (batch, seq_len, seq_len)                # Combined score        scores = (content_score + rel_score) / (self.d_model ** 0.5)        weights = F.softmax(scores, dim=-1)                # Value with relative position        rel_v = self.rel_emb_v(rel_indices)  # (seq_len, seq_len, d_model)                # Standard value aggregation + relative value contribution        content_out = weights @ V  # (batch, seq_len, d_model)        rel_out = torch.einsum('bij,ijd->bid', weights, rel_v)                return content_out + rel_out, weights, scores# Test ittorch.manual_seed(42)d_model = 32seq_len = 10max_rel_dist = 4model = ShawRelativeAttention(d_model, max_relative_distance=max_rel_dist)x = torch.randn(1, seq_len, d_model)output, weights, scores = model(x)print(f"Input shape:  {x.shape}")print(f"Output shape: {output.shape}")print(f"Relative distance range: [{-max_rel_dist}, {max_rel_dist}]")print(f"Number of relative position embeddings: {2 * max_rel_dist + 1}")print(f"\nRelative distance matrix (clipped to ±{max_rel_dist}):")rel_idx = model._get_relative_indices(seq_len) - max_rel_distprint(rel_idx.numpy())

### Visualizing: content-only vs content+relative scoresLet's see how the relative position bias shapes the attention pattern:

In [ ]:
# Decompose the attention score into content and relative partswith torch.no_grad():    Q = model.W_q(x)    K = model.W_k(x)        content_score = (Q @ K.transpose(-2, -1)) / (d_model ** 0.5)        rel_indices = model._get_relative_indices(seq_len)    rel_k = model.rel_emb_k(rel_indices)    rel_score = torch.einsum('bid,ijd->bij', Q, rel_k) / (d_model ** 0.5)        full_score = content_score + rel_scorefig, axes = plt.subplots(1, 3, figsize=(16, 4))titles = ['Content Score (q·k)', 'Relative Position Bias (q·r)', 'Full Score (sum)']matrices = [content_score[0], rel_score[0], full_score[0]]for ax, title, mat in zip(axes, titles, matrices):    vmax = mat.abs().max().item()    im = ax.imshow(mat.numpy(), cmap='RdBu_r', aspect='auto', vmin=-vmax, vmax=vmax)    ax.set_title(title, fontsize=11)    ax.set_xlabel('Key position')    ax.set_ylabel('Query position')    plt.colorbar(im, ax=ax, shrink=0.8)plt.suptitle('Shaw et al.: Content + Relative Position = Full Attention Score', fontsize=13, y=1.03)plt.tight_layout()plt.show()print("The relative position bias adds a structured pattern (diagonal bands)")print("that encourages attending to nearby tokens — regardless of absolute position.")

## Part 3: Transformer-XL (Dai et al., 2019) — Rethinking Relative AttentionTransformer-XL made two major contributions:1. A cleaner decomposition of relative attention2. Segment-level recurrence for processing long documents### The attention score decompositionIn standard absolute PE attention, the score between positions `i` and `j` is:$$(e_i + p_i)W_q \cdot (e_j + p_j)W_k$$Which expands to four terms (as we saw in the sinusoidal notebook):$$= \underbrace{e_i W_q \cdot e_j W_k}_{(a)} + \underbrace{e_i W_q \cdot p_j W_k}_{(b)} + \underbrace{p_i W_q \cdot e_j W_k}_{(c)} + \underbrace{p_i W_q \cdot p_j W_k}_{(d)}$$Transformer-XL replaces absolute positions $p_i, p_j$ with relative position encoding $R_{i-j}$and introduces two learnable bias vectors $u$ and $v$:$$\text{score}(i, j) = \underbrace{e_i W_q \cdot e_j W_k}_{\text{(a) content-content}} + \underbrace{e_i W_q \cdot R_{i-j} W_r}_{\text{(b) content-position}} + \underbrace{u \cdot e_j W_k}_{\text{(c) global content bias}} + \underbrace{v \cdot R_{i-j} W_r}_{\text{(d) global position bias}}$$Key changes from absolute PE:- Terms (b) and (d): absolute $p_j$ replaced with relative $R_{i-j}$- Term (c): $p_i W_q$ replaced with learnable vector $u$ (same for all query positions)- Term (d): $p_i W_q$ replaced with learnable vector $v$ (same for all query positions)- Separate weight matrix $W_r$ for position (instead of reusing $W_k$)Why $u$ and $v$ instead of $p_i W_q$?- The query's absolute position shouldn't matter for relative attention- $u$ learns a global "content importance" bias- $v$ learns a global "distance importance" bias

In [ ]:
class TransformerXLRelativeAttention(nn.Module):    """    Transformer-XL style relative attention with the 4-term decomposition.    """    def __init__(self, d_model, max_relative_distance=16):        super().__init__()        self.d_model = d_model                self.W_q = nn.Linear(d_model, d_model, bias=False)        self.W_k = nn.Linear(d_model, d_model, bias=False)        self.W_v = nn.Linear(d_model, d_model, bias=False)        self.W_r = nn.Linear(d_model, d_model, bias=False)  # separate matrix for position                # Global bias vectors (replace p_i * W_q)        self.u = nn.Parameter(torch.randn(d_model) * 0.02)  # content bias        self.v = nn.Parameter(torch.randn(d_model) * 0.02)  # position bias                # Sinusoidal relative position encoding (not learned — like original transformer)        self.max_rel = max_relative_distance        R = self._sinusoidal_relative_encoding(max_relative_distance, d_model)        self.register_buffer('R', R)        def _sinusoidal_relative_encoding(self, max_dist, d_model):        """Generate sinusoidal encoding for relative distances [-max_dist, max_dist]."""        positions = torch.arange(-max_dist, max_dist + 1, dtype=torch.float)        pe = torch.zeros(len(positions), d_model)        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float) * (-np.log(10000.0) / d_model))        pe[:, 0::2] = torch.sin(positions.unsqueeze(1) * div_term)        pe[:, 1::2] = torch.cos(positions.unsqueeze(1) * div_term)        return pe  # (2*max_dist+1, d_model)        def forward(self, x):        batch, seq_len, _ = x.shape                Q = self.W_q(x)  # (batch, seq_len, d_model)        K = self.W_k(x)        V = self.W_v(x)                # Build relative distance matrix and look up R embeddings        dists = torch.arange(seq_len).unsqueeze(0) - torch.arange(seq_len).unsqueeze(1)        dists = dists.clamp(-self.max_rel, self.max_rel) + self.max_rel  # shift to indices        R_ij = self.W_r(self.R[dists])  # (seq_len, seq_len, d_model)                # Term (a): content-content        term_a = Q @ K.transpose(-2, -1)                # Term (b): content-position  (q_i · R_{i-j})        term_b = torch.einsum('bid,ijd->bij', Q, R_ij)                # Term (c): global content bias  (u · k_j)        term_c = torch.einsum('d,bjd->bj', self.u, K).unsqueeze(1).expand(-1, seq_len, -1)                # Term (d): global position bias  (v · R_{i-j})        term_d = torch.einsum('d,ijd->ij', self.v, R_ij).unsqueeze(0).expand(batch, -1, -1)                scores = (term_a + term_b + term_c + term_d) / (self.d_model ** 0.5)        weights = F.softmax(scores, dim=-1)        output = weights @ V                return output, weights, {            'term_a': term_a[0].detach(),            'term_b': term_b[0].detach(),            'term_c': term_c[0].detach(),            'term_d': term_d[0].detach()        }torch.manual_seed(42)d_model = 32seq_len = 12model_xl = TransformerXLRelativeAttention(d_model, max_relative_distance=8)x = torch.randn(1, seq_len, d_model)output, weights, terms = model_xl(x)print(f"Input:  {x.shape}")print(f"Output: {output.shape}")print(f"\nThe 4 terms of Transformer-XL attention:")for name, desc in [('term_a', 'content↔content'), ('term_b', 'content↔position'),                    ('term_c', 'global content bias'), ('term_d', 'global position bias')]:    print(f"  {name} ({desc}): shape {terms[name].shape}, "          f"mean={terms[name].mean():.4f}, std={terms[name].std():.4f}")

### Visualizing the 4 terms of Transformer-XL attention

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(22, 4))titles = ['(a) Content↔Content', '(b) Content↔Position',           '(c) Global Content Bias', '(d) Global Position Bias', 'Full Score']full = terms['term_a'] + terms['term_b'] + terms['term_c'] + terms['term_d']matrices = [terms['term_a'], terms['term_b'], terms['term_c'], terms['term_d'], full]for ax, title, mat in zip(axes, titles, matrices):    vmax = mat.abs().max().item()    im = ax.imshow(mat.numpy(), cmap='RdBu_r', aspect='auto', vmin=-vmax, vmax=vmax)    ax.set_title(title, fontsize=9)    ax.set_xlabel('Key pos')    ax.set_ylabel('Query pos')    plt.colorbar(im, ax=ax, shrink=0.7)plt.suptitle('Transformer-XL: 4-Term Attention Decomposition', fontsize=13, y=1.05)plt.tight_layout()plt.show()print("Key observations:")print("  (a) Content↔Content: depends on what the tokens ARE (semantic)")print("  (b) Content↔Position: 'does this content care about that distance?'")print("  (c) Global Content Bias: 'which tokens are globally important?' (same for all queries)")print("  (d) Global Position Bias: 'which distances are globally preferred?' (diagonal bands)")print()print("Notice (d) has clear diagonal structure — it's a pure distance preference")print("that's the SAME regardless of what tokens are at those positions.")

## Part 4: Translation Invariance — The Key AdvantageThe whole point of relative PE: the attention between two tokens should depend on their**distance**, not their **absolute positions**.Let's verify: if we shift the entire sequence, do the attention patterns stay the same?

In [ ]:
torch.manual_seed(42)d_model = 32# Create a short "sentence" of 5 tokenstokens = torch.randn(1, 5, d_model)# Pad it at different starting positions within a longer sequencedef embed_at_position(tokens, start_pos, total_len, d_model):    """Place tokens starting at start_pos within a longer sequence of zeros."""    x = torch.zeros(1, total_len, d_model)    x[0, start_pos:start_pos + tokens.shape[1]] = tokens[0]    return xtotal_len = 16# Same tokens at position 0 and position 8x_pos0 = embed_at_position(tokens, 0, total_len, d_model)x_pos8 = embed_at_position(tokens, 8, total_len, d_model)# Run through relative attentionmodel_rel = TransformerXLRelativeAttention(d_model, max_relative_distance=8)_, weights_pos0, _ = model_rel(x_pos0)_, weights_pos8, _ = model_rel(x_pos8)# Extract the 5x5 attention submatrix for our tokensattn_at_0 = weights_pos0[0, 0:5, 0:5].detach()attn_at_8 = weights_pos8[0, 8:13, 8:13].detach()# Compare with absolute PE (standard attention + learned PE)class AbsoluteAttention(nn.Module):    def __init__(self, d_model, max_len):        super().__init__()        self.W_q = nn.Linear(d_model, d_model, bias=False)        self.W_k = nn.Linear(d_model, d_model, bias=False)        self.pos_emb = nn.Embedding(max_len, d_model)        self.d_model = d_model        def forward(self, x):        seq_len = x.shape[1]        pos = self.pos_emb(torch.arange(seq_len))        x = x + pos.unsqueeze(0)        Q = self.W_q(x)        K = self.W_k(x)        scores = Q @ K.transpose(-2, -1) / (self.d_model ** 0.5)        return F.softmax(scores, dim=-1)model_abs = AbsoluteAttention(d_model, total_len)abs_weights_pos0 = model_abs(x_pos0)abs_weights_pos8 = model_abs(x_pos8)abs_attn_at_0 = abs_weights_pos0[0, 0:5, 0:5].detach()abs_attn_at_8 = abs_weights_pos8[0, 8:13, 8:13].detach()fig, axes = plt.subplots(2, 3, figsize=(15, 9))# Row 1: Relative PEim0 = axes[0,0].imshow(attn_at_0.numpy(), cmap='Blues', vmin=0)axes[0,0].set_title('Relative PE: tokens at pos 0-4')im1 = axes[0,1].imshow(attn_at_8.numpy(), cmap='Blues', vmin=0)axes[0,1].set_title('Relative PE: tokens at pos 8-12')diff_rel = (attn_at_0 - attn_at_8).abs()im2 = axes[0,2].imshow(diff_rel.numpy(), cmap='Reds', vmin=0)axes[0,2].set_title(f'Difference (max={diff_rel.max():.4f})')# Row 2: Absolute PEim3 = axes[1,0].imshow(abs_attn_at_0.numpy(), cmap='Blues', vmin=0)axes[1,0].set_title('Absolute PE: tokens at pos 0-4')im4 = axes[1,1].imshow(abs_attn_at_8.numpy(), cmap='Blues', vmin=0)axes[1,1].set_title('Absolute PE: tokens at pos 8-12')diff_abs = (abs_attn_at_0 - abs_attn_at_8).abs()im5 = axes[1,2].imshow(diff_abs.numpy(), cmap='Reds', vmin=0)axes[1,2].set_title(f'Difference (max={diff_abs.max():.4f})')for row, label in zip(axes, ['RELATIVE PE', 'ABSOLUTE PE']):    row[0].set_ylabel(label, fontsize=12, fontweight='bold')for ax_row in axes:    for ax in ax_row:        plt.colorbar(ax.images[0], ax=ax, shrink=0.7)plt.suptitle('Translation Invariance Test: Same tokens at different positions', fontsize=13, y=1.02)plt.tight_layout()plt.show()print("RELATIVE PE: attention patterns are nearly identical regardless of position")print(f"  Max difference: {diff_rel.max():.6f}")print()print("ABSOLUTE PE: attention patterns CHANGE when tokens move to different positions")print(f"  Max difference: {diff_abs.max():.6f}")print()print("This is translation invariance — the key advantage of relative PE.")print("'cat sat on mat' should have the same internal attention whether it starts")print("at position 0 or position 100.")

## Part 5: The Distance Clipping EffectShaw et al. clip relative distances to $[-k, k]$. This means the model treats"50 tokens away" the same as "100 tokens away" (both become distance $k$).This is actually a reasonable inductive bias for many NLP tasks — very distant tokensrarely have precise positional relationships. But it limits the model's ability toreason about long-range structure.Let's visualize how clipping affects the position information:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))seq_len = 32for ax, k in zip(axes, [4, 8, 16]):    dists = torch.arange(seq_len).unsqueeze(0) - torch.arange(seq_len).unsqueeze(1)    clipped = dists.clamp(-k, k)        im = ax.imshow(clipped.numpy(), cmap='RdBu_r', aspect='auto')    ax.set_title(f'Clipped distances (k={k})', fontsize=11)    ax.set_xlabel('Key position')    ax.set_ylabel('Query position')    plt.colorbar(im, ax=ax, shrink=0.8)plt.suptitle('Effect of Distance Clipping: Larger k = More Position Resolution', fontsize=13, y=1.03)plt.tight_layout()plt.show()print("With k=4: positions more than 4 apart are indistinguishable (flat color in corners)")print("With k=16: much more position resolution, but still limited")print()print("The flat regions mean the model CANNOT distinguish between those distances.")print("This is a fundamental limitation of clipped relative PE.")

## Part 6: Limitations and the Path Forward| Aspect | Absolute PE | Shaw Relative | Transformer-XL ||--------|------------|---------------|----------------|| Encodes | "I am at position 47" | "We are 3 apart" | "We are 3 apart" + global biases || Translation invariant | ❌ | ✅ | ✅ || Extrapolation | Degrades / crashes | Better (distances generalize) | Better + recurrence || Complexity | O(1) per position | O(n²) relative lookups | O(n²) + recurrence overhead || Long-range | Limited by training length | Limited by clipping distance k | Better via recurrence |### What relative PE got right:- The fundamental insight that **relative distance > absolute position**- Translation invariance- Better generalization to unseen sequence lengths### What it still got wrong:- Clipping loses information about long-range distances- The learned relative embeddings are still a finite table- Transformer-XL's recurrence adds sequential dependency (hurts parallelism)- Complex implementation with multiple terms and special matrices### The path to RoPE:RoPE takes the relative position insight and implements it more elegantly:- No separate position embeddings or bias terms- No clipping — every distance is uniquely encoded- Relative position falls out naturally from the math (rotation subtraction)- Simple to implement, efficient to compute- Compatible with KV-cache for fast inferenceNext notebook: **RoPE — Rotary Position Embedding** 🚀